# PV Defect Detector — Evaluation

Evaluate the trained model on the held-out test set.

**Prerequisites:** Run `task training:train` and `task training:export` before this notebook.

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tflite_runtime.interpreter import Interpreter

DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')

LABELS = ['hotspot', 'bypass_diode_failure', 'soiling', 'multi_hotspot', 'shadowing', 'delamination']
COLORS = ['#e74c3c', '#e67e22', '#f1c40f', '#c0392b', '#3498db', '#9b59b6']

In [ ]:
# Load test annotations
with open(DATA_DIR / 'test.json') as f:
    test_coco = json.load(f)

print(f"Test images: {len(test_coco['images'])}")
print(f"Test annotations: {len(test_coco['annotations'])}")

from collections import Counter
class_counts = Counter(a['category_id'] for a in test_coco['annotations'])
for cid, count in sorted(class_counts.items()):
    print(f"  {LABELS[cid]}: {count}")

In [ ]:
# Load float model for evaluation
interpreter = Interpreter(model_path=str(MODELS_DIR / 'pv_detector_float.tflite'))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
_, h, w, _ = input_details[0]['shape']
print(f'Input shape: {h}x{w}')

In [ ]:
# Run inference on all test images
predictions = []
images_dir = DATA_DIR / 'images'

for img_info in test_coco['images']:
    img = cv2.imread(str(images_dir / img_info['file_name']))
    if img is None:
        continue
    resized = cv2.resize(img, (w, h))
    inp = np.expand_dims(resized, axis=0).astype(np.float32) / 255.0
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    boxes = interpreter.get_tensor(output_details[0]['index'])[0]
    class_ids = interpreter.get_tensor(output_details[1]['index'])[0]
    scores = interpreter.get_tensor(output_details[2]['index'])[0]
    num = int(interpreter.get_tensor(output_details[3]['index'])[0])
    predictions.append({'image_id': img_info['id'], 'boxes': boxes[:num],
                         'class_ids': class_ids[:num].astype(int), 'scores': scores[:num]})

print(f'Inference complete on {len(predictions)} images')

In [ ]:
# Precision / Recall per class at threshold=0.35
THRESHOLD = 0.35
IOU_THRESH = 0.5

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

gt_by_img = {}
for ann in test_coco['annotations']:
    gt_by_img.setdefault(ann['image_id'], []).append(ann)

tp = [0] * 6; fp = [0] * 6; fn_count = [0] * 6

for pred in predictions:
    gt_anns = gt_by_img.get(pred['image_id'], [])
    matched = set()
    for i, (score, cls_id) in enumerate(zip(pred['scores'], pred['class_ids'])):
        if score < THRESHOLD:
            continue
        box = pred['boxes'][i]  # [ymin, xmin, ymax, xmax] normalised
        best_iou, best_j = 0, -1
        for j, gt in enumerate(gt_anns):
            if j in matched or gt['category_id'] != cls_id:
                continue
            x, y, bw, bh = gt['bbox']
            img_w = gt.get('w', 640); img_h = gt.get('h', 480)
            gt_norm = [y/img_h, x/img_w, (y+bh)/img_h, (x+bw)/img_w]
            this_iou = iou(box, gt_norm)
            if this_iou > best_iou:
                best_iou, best_j = this_iou, j
        if best_iou >= IOU_THRESH:
            tp[cls_id] += 1; matched.add(best_j)
        else:
            fp[cls_id] += 1
    for j, gt in enumerate(gt_anns):
        if j not in matched:
            fn_count[gt['category_id']] += 1

print(f"{'Class':<25} {'TP':>5} {'FP':>5} {'FN':>5} {'Precision':>10} {'Recall':>8}")
print('-' * 65)
for i, label in enumerate(LABELS):
    prec = tp[i] / (tp[i] + fp[i] + 1e-6)
    rec  = tp[i] / (tp[i] + fn_count[i] + 1e-6)
    print(f'{label:<25} {tp[i]:>5} {fp[i]:>5} {fn_count[i]:>5} {prec:>10.1%} {rec:>8.1%}')

In [ ]:
# Visualise sample predictions (first 6 test images)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, pred in zip(axes.flat, predictions[:6]):
    img_info = next(i for i in test_coco['images'] if i['id'] == pred['image_id'])
    img = cv2.imread(str(images_dir / img_info['file_name']))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ih, iw = img_rgb.shape[:2]
    for score, cls_id, box in zip(pred['scores'], pred['class_ids'], pred['boxes']):
        if score < THRESHOLD:
            continue
        y1, x1, y2, x2 = int(box[0]*ih), int(box[1]*iw), int(box[2]*ih), int(box[3]*iw)
        color = tuple(int(c*255) for c in plt.cm.tab10(cls_id)[:3])
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img_rgb, f'{LABELS[cls_id]} {score:.0%}', (x1, max(y1-5,0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    ax.imshow(img_rgb)
    ax.set_title(img_info['file_name'])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Quantization accuracy delta: float vs int8
interp_quant = Interpreter(model_path=str(MODELS_DIR / 'pv_detector_quant.tflite'))
interp_quant.allocate_tensors()
input_q = interp_quant.get_input_details()
output_q = interp_quant.get_output_details()
_, hq, wq, _ = input_q[0]['shape']

quant_preds = []
for img_info in test_coco['images']:
    img = cv2.imread(str(images_dir / img_info['file_name']))
    if img is None:
        continue
    resized = cv2.resize(img, (wq, hq))
    inp = np.expand_dims(resized, axis=0)
    # int8 model expects uint8 input (no /255 normalisation)
    interp_quant.set_tensor(input_q[0]['index'], inp)
    interp_quant.invoke()
    boxes = interp_quant.get_tensor(output_q[0]['index'])[0]
    class_ids = interp_quant.get_tensor(output_q[1]['index'])[0]
    scores = interp_quant.get_tensor(output_q[2]['index'])[0]
    num = int(interp_quant.get_tensor(output_q[3]['index'])[0])
    quant_preds.append({'image_id': img_info['id'], 'boxes': boxes[:num],
                         'class_ids': class_ids[:num].astype(int), 'scores': scores[:num]})

# Count detections above threshold for both models
float_dets = sum(sum(1 for s in p['scores'] if s >= THRESHOLD) for p in predictions)
quant_dets = sum(sum(1 for s in p['scores'] if s >= THRESHOLD) for p in quant_preds)
print(f'Float model detections (≥{THRESHOLD:.0%}): {float_dets}')
print(f'Quant model detections (≥{THRESHOLD:.0%}): {quant_dets}')
print(f'Delta: {abs(float_dets - quant_dets)} ({abs(float_dets-quant_dets)/(float_dets+1e-6):.1%})')